Se lleva a cabo el montaje de la unidad de Google Drive con el objetivo de acceder a los archivos necesarios para el desarrollo del presente trabajo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Instalación del paquete `datasets`

Se instala el paquete `datasets` de Hugging Face, que proporciona acceso a una gran colección de conjuntos de datos preparados para tareas comunes de procesamiento de lenguaje natural, visión por computador y otros problemas de aprendizaje automático.


In [ ]:
!pip install datasets --quiet



### Importación de librerías necesarias

Se importan las librerías y módulos esenciales para el desarrollo del proyecto, que incluye:

- Manejo de archivos y datos (`os`, `pandas`, `joblib`).
- Preprocesamiento de etiquetas y métricas de evaluación con `scikit-learn`.
- Soporte de GPU y entrenamiento con `PyTorch`.
- Manipulación de conjuntos de datos con `datasets` de Hugging Face.
- Carga de modelos preentrenados, tokenización y entrenamiento utilizando la librería `transformers`.
- Acceso a Google Drive para lectura y escritura de archivos dentro del entorno de Google Colab.


In [ ]:
# Acceso a sistema de archivos y utilidades
import os
import joblib

# Manipulación de datos
import pandas as pd

# Soporte para entrenamiento en GPU
import torch

# Preprocesamiento y métricas de evaluación
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# Manejo de conjuntos de datos con Hugging Face
from datasets import Dataset

# Modelos preentrenados y utilidades de entrenamiento
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)




### Carga de datos de clasificación de productos

Se procede a la lectura del archivo CSV que contiene las correspondencias de productos hasta el año 2022. El archivo está codificado en `latin1` y utiliza `;` como separador de columnas. A continuación se muestran las primeras filas del conjunto de datos para verificar que se ha cargado correctamente.


In [ ]:
df_product_classification_2022 = pd.read_csv(
    '/content/drive/MyDrive/Productos/datosSOY_SUPER/Correspondencias hasta 2022.csv',
    encoding='latin1',
    sep=';'
)

df_product_classification_2022.head()

,DESC_ART,CATEGORIA,DESC_PROD
0,Azúcar granulado Tate + Lyle,"Panadería, Pastelería y Repostería Azúcar y Ed...",Azúcar blanco_WKB
1,Leche infantil Blemil Plus 2 AE Blemil,Bebés y Niños Alimentación Infantil Leche infa...,Leche en polvo para bebés_WKB
2,Crunchy Muesli ATKINS DAY BREAK,Cereales y Galletas Cereales ###Clásicos,Muesli crujiente_WKB
3,Mandarina a granel,Frescos y Charcutería Frutas Cítricos,Mandarinas_NE
4,Yogurt natural artesanal ecológico purnatur Pu...,Lácteos y Huevos Yogur ###Sabores,Yogurt natural_WKB


### Entrenamiento con validación cruzada

Se entrena un modelo de lenguaje preentrenado seleccionado (por defecto, `PlanTL-GOB-ES/roberta-base-bne`) para la tarea de clasificación de productos en español. Se realiza validación cruzada con 4 folds, cargando y preprocesando los datos desde Google Drive.

Para cada iteración de la validación cruzada, se usan conjuntos previamente separados:
- Cada archivo `productos_parte_X_train.csv` contiene los datos de entrenamiento correspondientes al fold `X`, excluyendo el fold `X` como conjunto de validación.
- El archivo `productos_parte_X.csv` contiene el fold `X`, que se usa como conjunto de validación.

Por ejemplo:
- En la parte 1, se entrena con `productos_parte_1_train.csv` (que contiene los folds 2, 3 y 4) y se valida con `productos_parte_1.csv` (fold 1).
- En la parte 2, se entrena con los folds 1, 3 y 4 y se valida con el fold 2.
- Y así sucesivamente para los 4 folds.

Para cada fold se realiza:
- Lectura de los datos de entrenamiento y validación desde Google Drive.
- Codificación de las etiquetas con `LabelEncoder`.
- Concatenación del texto del artículo con su categoría como entrada del modelo.
- Tokenización del texto usando el modelo seleccionado.
- Entrenamiento utilizando el `Trainer` de Hugging Face.
- Evaluación de precisión (`accuracy`), `recall`, `f1-score` y generación de la matriz de confusión.
- Guardado de:
  - Métricas del fold.
  - Modelo entrenado.
  - Tokenizador.
  - Codificador de etiquetas (`LabelEncoder`).



In [ ]:
# 🧠 Modelo a usar
nickname = "maria"  # Cambia aquí si quieres probar otro modelo

model_name = {
    "bert": "bert-base-uncased",
    "distilbert": "distilbert-base-uncased",
    "minilm": "nreimers/MiniLM-L6-H384-uncased",
    "mobilebert": "google/mobilebert-uncased",
    "tinybert": "huawei-noah/TinyBERT_General_4L_312D",
    "albert": "albert-base-v2",
    "electra": "google/electra-small-discriminator",
    "roberta": "roberta-base",
    "xlmroberta": "xlm-roberta-base",
    "deberta": "microsoft/deberta-v3-small",
    "camembert": "camembert-base",
    "gpt2": "gpt2",
    "xlmroberta_large": "xlm-roberta-large",
    "beto": "dccuchile/bert-base-spanish-wwm-uncased",
    "distilbeto": "dccuchile/distilbert-base-spanish-uncased",
    "maria": "PlanTL-GOB-ES/roberta-base-bne"
}[nickname]




# 📄 Rutas de entrenamiento y validación
train_paths = [
    "/content/drive/MyDrive/Productos/datosSOY_SUPER/validacion_cruzada/productos_parte_1_train.csv",
    "/content/drive/MyDrive/Productos/datosSOY_SUPER/validacion_cruzada/productos_parte_2_train.csv",
    "/content/drive/MyDrive/Productos/datosSOY_SUPER/validacion_cruzada/productos_parte_3_train.csv",
    "/content/drive/MyDrive/Productos/datosSOY_SUPER/validacion_cruzada/productos_parte_4_train.csv"
]

test_paths = [
    "/content/drive/MyDrive/Productos/datosSOY_SUPER/validacion_cruzada/productos_parte_1.csv",
    "/content/drive/MyDrive/Productos/datosSOY_SUPER/validacion_cruzada/productos_parte_2.csv",
    "/content/drive/MyDrive/Productos/datosSOY_SUPER/validacion_cruzada/productos_parte_3.csv",
    "/content/drive/MyDrive/Productos/datosSOY_SUPER/validacion_cruzada/productos_parte_4.csv"
]

# 🧰 Funciones para evaluación y guardado de métricas
def compute_metrics_factory(fold_id):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = logits.argmax(-1)

        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average="weighted", zero_division=0
        )
        accuracy = accuracy_score(labels, predictions)

        metrics_dir = f"/content/drive/MyDrive/Productos/metrics_val_cruzada/{nickname}_fold_{fold_id}"
        os.makedirs(metrics_dir, exist_ok=True)

        cm = confusion_matrix(labels, predictions)
        pd.DataFrame(cm).to_csv(os.path.join(metrics_dir, "confusion_matrix.csv"), index=False)

        metrics = {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }
        pd.DataFrame([metrics]).to_csv(os.path.join(metrics_dir, "metrics.csv"), index=False)

        return metrics
    return compute_metrics

# 📊 Entrenamiento por fold
for i in range(len(train_paths)):
    print(f"\n🔁 Fold {i+1} - Entrenando con {model_name}")

    train_df = pd.read_csv(train_paths[i])
    test_df = pd.read_csv(test_paths[i])

    label_encoder = LabelEncoder()
    train_df["label"] = label_encoder.fit_transform(train_df["DESC_PROD"])
    test_df["label"] = label_encoder.transform(test_df["DESC_PROD"])
    num_labels = len(label_encoder.classes_)

    train_df["text"] = train_df["DESC_ART"] + " | " + train_df["CATEGORIA"]
    test_df["text"] = test_df["DESC_ART"] + " | " + test_df["CATEGORIA"]

    train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
    test_dataset = Dataset.from_pandas(test_df[["text", "label"]])

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    def tokenize(example):
        return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

    tokenized_train = train_dataset.map(tokenize)
    tokenized_test = test_dataset.map(tokenize)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

    # Carpeta de checkpoints
    output_dir = f"/content/drive/MyDrive/checkpoints_val_cruzada/{nickname}_fold{i+1}"
    os.makedirs(output_dir, exist_ok=True)

    # Buscar último checkpoint
    last_checkpoint = None
    if os.path.isdir(output_dir):
        checkpoints = [os.path.join(output_dir, d) for d in os.listdir(output_dir) if d.startswith("checkpoint")]
        if checkpoints:
            last_checkpoint = max(checkpoints, key=os.path.getctime)
            print(f"🔄 Reanudando desde checkpoint: {last_checkpoint}")

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=8,
        num_train_epochs=4,
        logging_dir=os.path.join(output_dir, "logs"),
        logging_steps=10,
        save_steps=50000,
        save_total_limit=2,
        load_best_model_at_end=False,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics_factory(i + 1)
    )

    trainer.train(resume_from_checkpoint=last_checkpoint)

    # Evaluar
    eval_results = trainer.evaluate()
    print(f"\n📊 Resultados del Fold {i+1}:")
    for k, v in eval_results.items():
        print(f"  {k}: {v:.4f}")

    # Guardar modelo final
    final_model_dir = f"/content/drive/MyDrive/modelos_validados_crossval/{nickname}_fold{i+1}"
    os.makedirs(final_model_dir, exist_ok=True)

    trainer.save_model(final_model_dir)
    tokenizer.save_pretrained(final_model_dir)
    joblib.dump(label_encoder, os.path.join(final_model_dir, "label_encoder.pkl"))
    print(f"✅ Modelo guardado en: {final_model_dir}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🔁 Fold 1 - Entrenando con PlanTL-GOB-ES/roberta-base-bne


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/851k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/509k [00:00<?, ?B/s]

...

Map:   0%|          | 0/19715 [00:00<?, ? examples/s]

Map:   0%|          | 0/6419 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at PlanTL-GOB-ES/roberta-base-bne and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-5-39eff96e84f2>:137: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
10,5.727200
20,5.603500
30,5.491600
40,5.399700
50,5.133600
60,4.996100
70,4.968900
80,4.930200
90,4.780800
100,4.792800



📊 Resultados del Fold 4:
  eval_loss: 0.2948
  eval_accuracy: 0.9181
  eval_precision: 0.9170
  eval_recall: 0.9181
  eval_f1: 0.9097
  eval_runtime: 42.3599
  eval_samples_per_second: 151.5350
  eval_steps_per_second: 18.9570
  epoch: 4.0000
✅ Modelo guardado en: /content/drive/MyDrive/modelos_validados_crossval/maria_fold4


### Entrenamiento individual del modelo y fold seleccionados

Este bloque permite entrenar de forma individual cualquier modelo preentrenado especificado en el diccionario `model_name`, aplicándolo sobre una partición concreta de los datos (por ejemplo, el fold 1).

A diferencia del entrenamiento en bucle para todos los folds, aquí el usuario puede:

- Cambiar fácilmente el modelo modificando el valor de `nickname`.
- Seleccionar cualquier fold ajustando las rutas `train_path` y `test_path`.

El proceso incluye:
- Codificación de etiquetas con `LabelEncoder`.
- Preparación del texto combinando descripción del producto y categoría.
- Tokenización mediante el tokenizador del modelo seleccionado.
- Entrenamiento supervisado con `Trainer`.
- Evaluación automática con métricas de `scikit-learn`.
- Guardado del modelo, tokenizador, codificador de etiquetas y métricas en Google Drive.

Esto permite realizar pruebas y comparaciones personalizadas modelo a modelo y fold a fold.



In [ ]:

# 🧠 Modelo a usar
nickname = "deberta"  # Cambia aquí si quieres probar otro modelo

model_name = {
    "bert": "bert-base-uncased",
    "distilbert": "distilbert-base-uncased",
    "minilm": "nreimers/MiniLM-L6-H384-uncased",
    "mobilebert": "google/mobilebert-uncased",
    "tinybert": "huawei-noah/TinyBERT_General_4L_312D",
    "albert": "albert-base-v2",
    "electra": "google/electra-small-discriminator",
    "roberta": "roberta-base",
    "xlmroberta": "xlm-roberta-base",
    "deberta": "microsoft/deberta-v3-small",
    "camembert": "camembert-base",
    "gpt2": "gpt2",
    "xlmroberta_large": "xlm-roberta-large",
    "beto": "dccuchile/bert-base-spanish-wwm-uncased",
    "distilbeto": "dccuchile/distilbert-base-spanish-uncased",
    "maria": "PlanTL-GOB-ES/roberta-base-bne"
}[nickname]


# 📄 Rutas de entrenamiento y validación (solo cuarta parte)
train_path = "/content/drive/MyDrive/Productos/datosSOY_SUPER/validacion_cruzada/productos_parte_1_train.csv"
test_path = "/content/drive/MyDrive/Productos/datosSOY_SUPER/validacion_cruzada/productos_parte_1.csv"

# 🧰 Funciones para evaluación y guardado de métricas
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0
    )
    accuracy = accuracy_score(labels, predictions)

    metrics_dir = f"/content/drive/MyDrive/Productos/metrics_val_cruzada/{nickname}_fold_1"
    os.makedirs(metrics_dir, exist_ok=True)

    cm = confusion_matrix(labels, predictions)
    pd.DataFrame(cm).to_csv(os.path.join(metrics_dir, "confusion_matrix.csv"), index=False)

    metrics = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }
    pd.DataFrame([metrics]).to_csv(os.path.join(metrics_dir, "metrics.csv"), index=False)

    return metrics

# 📊 Entrenamiento para el cuarto fold
print(f"\n🔁 Fold 4 - Entrenando con {model_name}")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

label_encoder = LabelEncoder()
train_df["label"] = label_encoder.fit_transform(train_df["DESC_PROD"])
test_df["label"] = label_encoder.transform(test_df["DESC_PROD"])
num_labels = len(label_encoder.classes_)

train_df["text"] = train_df["DESC_ART"] + " | " + train_df["CATEGORIA"]
test_df["text"] = test_df["DESC_ART"] + " | " + test_df["CATEGORIA"]

train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["text", "label"]])

tokenizer = AutoTokenizer.from_pretrained(model_name)
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

tokenized_train = train_dataset.map(tokenize)
tokenized_test = test_dataset.map(tokenize)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# Carpeta de checkpoints
output_dir = f"/content/drive/MyDrive/checkpoints_val_cruzada/{nickname}_fold1"
os.makedirs(output_dir, exist_ok=True)

# Buscar último checkpoint
last_checkpoint = None
if os.path.isdir(output_dir):
    checkpoints = [os.path.join(output_dir, d) for d in os.listdir(output_dir) if d.startswith("checkpoint")]
    if checkpoints:
        last_checkpoint = max(checkpoints, key=os.path.getctime)
        print(f"🔄 Reanudando desde checkpoint: {last_checkpoint}")

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=8,
    num_train_epochs=4,
    logging_dir=os.path.join(output_dir, "logs"),
    logging_steps=10,
    save_steps=10000,
    save_total_limit=2,
    load_best_model_at_end=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train(resume_from_checkpoint=last_checkpoint)

# Evaluar
eval_results = trainer.evaluate()
print(f"\n📊 Resultados del Fold 1:")
for k, v in eval_results.items():
    print(f"  {k}: {v:.4f}")

# Guardar modelo final
final_model_dir = f"/content/drive/MyDrive/modelos_validados_crossval/{nickname}_fold1"
os.makedirs(final_model_dir, exist_ok=True)

trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
joblib.dump(label_encoder, os.path.join(final_model_dir, "label_encoder.pkl"))
print(f"✅ Modelo guardado en: {final_model_dir}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🔁 Fold 4 - Entrenando con microsoft/deberta-v3-small


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-6-3309384833.py:121: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
10,5.738200
20,5.747100
30,5.712100
40,5.599400
50,5.478000
60,5.443200
70,5.495200
80,5.334800
90,5.379400
100,5.358000


Step,Training Loss
10,5.738200
20,5.747100
30,5.712100
40,5.599400
50,5.478000
60,5.443200
70,5.495200
80,5.334800
90,5.379400
100,5.358000


KeyboardInterrupt: 